# Agent Seed Extraction And Scene Tracking

This notebook runs full-video scene splitting, analyzes each scene for object seeds, sends those seeds to the SAM3 visual inference server, and renders scene-local tracking overlays.

In [ ]:
from pathlib import Path
import dotenv

dotenv.load_dotenv(dotenv.find_dotenv())

import httpx

from v2a_inspect.client import SAM3Client, VideoClient
from v2a_inspect.preprocessing import (
    analyze_initial_scenes,
    preprocess_video,
    sam3_tracking_video_path,
    track_initial_scenes_object_seeds,
)
from v2a_inspect.visualization import (
    display_image,
    display_video,
    render_keyframe_grid,
    render_scene_timeline,
    render_tracking_video,
    summarize_scenes,
    summarize_tracks,
)

## Config

By default, this runs every detected scene. Set `SELECTED_SCENE_INDEXES` or `MAX_SCENES_TO_PROCESS` to limit cost while tuning prompts and server memory use.

In [ ]:
SERVER_URL = "http://localhost:8080"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_VIDEO_PATH = (PROJECT_ROOT / "test.mp4").resolve()
OUTPUT_DIR = PROJECT_ROOT / "demo" / "outputs" / "agent_seed_tracking"
WORK_DIR = OUTPUT_DIR / "work"
SELECTED_SCENE_INDEXES = None
MAX_SCENES_TO_PROCESS = None
MAX_KEYFRAMES_PER_SCENE = 8

RUN_LLM_ANALYSIS = True
RUN_SERVER_REQUESTS = True
RUN_TRACKING = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
RAW_VIDEO_PATH

## Local Preprocessing

This stays local: normalize the raw video, create the 360p SAM3 tracking variant, detect initial scenes, and extract representative keyframes. NVENC is used by default for H.264 encoding.

In [ ]:
video_asset = preprocess_video(
    RAW_VIDEO_PATH,
    WORK_DIR,
    max_keyframes_per_scene=MAX_KEYFRAMES_PER_SCENE,
)
scene_indexes = (
    list(range(len(video_asset.initial_scenes)))
    if SELECTED_SCENE_INDEXES is None
    else list(SELECTED_SCENE_INDEXES)
)
if MAX_SCENES_TO_PROCESS is not None:
    scene_indexes = scene_indexes[:MAX_SCENES_TO_PROCESS]

{
    "raw_path": str(RAW_VIDEO_PATH),
    "prepared_path": str(video_asset.source_path),
    "sam3_tracking_path": None
    if video_asset.sam3_tracking_path is None
    else str(video_asset.sam3_tracking_path),
    "frame_count": video_asset.frame_count,
    "duration_sec": video_asset.duration_sec,
    "scene_count": len(video_asset.initial_scenes),
    "processing_scene_count": len(scene_indexes),
    "scene_indexes": scene_indexes,
}

In [ ]:
display_image(render_scene_timeline(video_asset))
display_image(render_keyframe_grid([video_asset.initial_scenes[index] for index in scene_indexes]))
summarize_scenes(video_asset)

## Seed Extraction

The local LLM analyzes each configured scene and produces `ObjectSeed`s. These seeds are later converted to SAM3 text prompts.

In [ ]:
if RUN_LLM_ANALYSIS:
    scenes_to_analyze = [video_asset.initial_scenes[index] for index in scene_indexes]
    analyzed_scenes = analyze_initial_scenes(scenes_to_analyze)
    updated_scenes = list(video_asset.initial_scenes)
    for scene_index, analyzed_scene in zip(scene_indexes, analyzed_scenes, strict=True):
        updated_scenes[scene_index] = analyzed_scene
    video_asset = video_asset.model_copy(update={"initial_scenes": updated_scenes})
else:
    print("Skipped. Set RUN_LLM_ANALYSIS = True to extract object seeds.")

In [ ]:
seed_rows = []
for scene_index in scene_indexes:
    scene = video_asset.initial_scenes[scene_index]
    if scene.initial_analysis is None:
        continue
    for object_seed in scene.initial_analysis.object_seeds:
        seed_rows.append(
            {
                "scene_index": scene_index,
                "label": object_seed.label,
                "tracking_prompt": object_seed.tracking_prompt,
                "notes": object_seed.notes,
            }
        )
seed_rows

## Upload And Track

The server is visual-inference-only. It receives the generated SAM3 tracking video plus scene-bounded SAM3 requests for the configured scenes.

In [ ]:
if RUN_SERVER_REQUESTS:
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=30.0) as http_client:
        health = await http_client.get("/healthz")
        health.raise_for_status()
        print(health.json())
else:
    print("Skipped. Set RUN_SERVER_REQUESTS = True to call the server.")

In [ ]:
tracking_video_path = sam3_tracking_video_path(video_asset)
video_id = None
if RUN_SERVER_REQUESTS:
    async with VideoClient(SERVER_URL) as video_client:
        upload_response = await video_client.upload(str(tracking_video_path))
        video_id = upload_response.video_id
{"video_id": video_id, "uploaded_path": str(tracking_video_path)}

In [ ]:
if RUN_SERVER_REQUESTS and RUN_TRACKING and video_id is not None:
    async with SAM3Client(SERVER_URL, timeout=1800.0) as sam_client:
        video_asset = await track_initial_scenes_object_seeds(
            video_asset,
            video_id=video_id,
            sam_client=sam_client,
            scene_indexes=scene_indexes,
        )
else:
    print("Skipped. Enable server requests and tracking after seeds are available.")

## Visualize Results

In [ ]:
for scene_index in scene_indexes:
    scene = video_asset.initial_scenes[scene_index]
    print({
        "scene_index": scene_index,
        "start_frame_index": scene.start_frame_index,
        "end_frame_index": scene.end_frame_index,
        "seed_count": 0 if scene.initial_analysis is None else len(scene.initial_analysis.object_seeds),
        "track_count": len(scene.scene_tracks),
    })
    if scene.scene_tracks:
        overlay_path = OUTPUT_DIR / f"scene_{scene_index:03d}_seed_tracks.mp4"
        render_tracking_video(
            video_asset.source_path,
            scene.scene_tracks,
            overlay_path,
            start_frame_index=scene.start_frame_index,
            end_frame_index=scene.end_frame_index,
        )
        display_video(overlay_path)
        print(summarize_tracks(scene.scene_tracks))

## Save Artifact

In [ ]:
asset_path = OUTPUT_DIR / "video_asset.json"
asset_path.write_text(video_asset.model_dump_json(indent=2), encoding="utf-8")
asset_path